# 01 · Exploração e diagnóstico dos dados brutos do Olist

> O objetivo dese notebook é entender a o estado dos CSVs. Cada problema achado aqui vira uma regra de limpeza

## O pipeline em uma imagem

```
 CSVs do Kaggle        RAW                 STAGING                 DW (esquema estrela)         Dashboard
 (data/raw/)   ──▶  cópia fiel   ──▶   limpeza + tipagem   ──▶   dimensões + fatos    ──▶   Streamlit
                    (notebook 02)       (notebook 03)             (notebook 04)               (notebook 06)
                                              │                          │
                                     staging.dq_issues            validações (notebook 05)
                               (registro de tudo que foi achado)
```

- **Raw**: os dados exatamente como vieram, sem alterar.
- **Staging**: os dados limpos, tipados e sem duplicatas. É onde os erros são tratados.
- **DW**: o modelo pronto para análise (dimensões e fatos).

## Roteiro deste notebook
1. Carregar os 9 CSVs e conhecer cada um
2. Como as tabelas se relacionam
3. Nulos e duplicatas (triagem)
4. **Diagnóstico por tema:** CEP · cidades · UF × CEP · geolocalização · categorias · números · datas · pagamentos · avaliações · integridade
5. Status final: o que precisamos tratar

In [2]:
# Configuração inicial: localizar a raiz do projeto e ajustar o diretório de trabalho,
# para que os caminhos relativos (config/, data/, src/) funcionem independentemente de onde
# o notebook seja executado (Jupyter local, VS Code ou Docker).
import logging
import os
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
logging.getLogger("streamlit").setLevel(logging.ERROR)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)


def find_project_root():
    current = Path.cwd().resolve()
    for p in [current] + list(current.parents):
        if (p / "src").exists() and (p / "data").exists() and (p / "config").exists():
            return p
    raise RuntimeError("Raiz do projeto não encontrada")


def project_path(*segments):
    return find_project_root().joinpath(*segments)


root = find_project_root()
os.chdir(root)
print(f"Diretório de trabalho: {root}")

Diretório de trabalho: C:\Users\user\Downloads\Códigos\olist-ecommerce-pipeline\Template


## 1. Carregando os CSVs

In [3]:
CSV_FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

# O `pd.read_csv` adivinha o tipo de cada coluna. Isso é a origem de vários problemas

tables = {}
for name, filename in CSV_FILES.items():
    path = project_path("data", "raw", filename)
    if path.exists():
        tables[name] = pd.read_csv(path)
    else:
        print(f"Faltando: {filename} - baixe via `python scripts/kaggle_ingest.py` ou insira manualmente em data/raw/")

print(f"{len(tables)} de {len(CSV_FILES)} tabelas carregadas")

9 de 9 tabelas carregadas


In [ ]:
overview = pd.DataFrame({
    "linhas": {n: len(d) for n, d in tables.items()},
    "colunas": {n: d.shape[1] for n, d in tables.items()},
    "memória (MB)": {n: round(d.memory_usage(deep=True).sum() / 1e6, 1) for n, d in tables.items()},
})
overview.loc["TOTAL"] = overview.sum()
overview

# `geolocation` tem 1 milhão de linhas, 2/3 do total

,linhas,colunas,memória (MB)
customers,99441.0,5.0,27.9
orders,99441.0,8.0,55.5
order_items,112650.0,7.0,37.7
order_payments,103886.0,5.0,17.0
order_reviews,99224.0,7.0,41.0
products,32951.0,9.0,6.6
sellers,3095.0,4.0,0.6
geolocation,1000163.0,5.0,135.7
category_translation,71.0,2.0,0.0
TOTAL,1550922.0,52.0,322.0


### Tipos que o pandas escolheu

In [5]:
for name, df in tables.items():
    print(f"── {name} ".ljust(60, "─"))
    print(df.dtypes.to_frame("tipo").T.to_string())

── customers ───────────────────────────────────────────────
     customer_id customer_unique_id customer_zip_code_prefix customer_city customer_state
tipo      object             object                    int64        object         object
── orders ──────────────────────────────────────────────────
     order_id customer_id order_status order_purchase_timestamp order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
tipo   object      object       object                   object            object                       object                        object                        object
── order_items ─────────────────────────────────────────────
     order_id order_item_id product_id seller_id shipping_limit_date    price freight_value
tipo   object         int64     object    object              object  float64       float64
── order_payments ──────────────────────────────────────────
     order_id payment_sequential payment_type paym

## 2. Como as tabelas se relacionam

```
customers ──(customer_id)──▶ orders ──(order_id)──▶ order_items ──(product_id)──▶ products ──(categoria)──▶ category_translation
                               │                        └────────(seller_id)────▶ sellers
                               ├──(order_id)──▶ order_payments
                               └──(order_id)──▶ order_reviews

customers / sellers ──(CEP)──▶ geolocation
```

In [7]:
# `customer_id` **muda a cada pedido**. A entidade é o `customer_unique_id`.

customers, orders, items = tables["customers"], tables["orders"], tables["order_items"]

print("Pedidos:                      ", f"{orders['order_id'].nunique():,}")
print("Itens de pedido:              ", f"{len(items):,}")
print("Pedidos com pelo menos 1 item:", f"{items['order_id'].nunique():,}")
print("customer_id distintos:        ", f"{customers['customer_id'].nunique():,}")
print("Pessoas reais (unique_id):    ", f"{customers['customer_unique_id'].nunique():,}")
print("Pessoas que compraram +1 vez: ", f"{(customers.groupby('customer_unique_id').size() > 1).sum():,}")


Pedidos:                       99,441
Itens de pedido:               112,650
Pedidos com pelo menos 1 item: 98,666
customer_id distintos:         99,441
Pessoas reais (unique_id):     96,096
Pessoas que compraram +1 vez:  2,997


## 3. Triagem: nulos e duplicatas

### 3.1 Valores nulos por coluna

In [8]:
null_report = pd.concat(
    {n: d.isna().sum().loc[lambda s: s > 0].to_frame("nulos").assign(pct=lambda x: (x["nulos"] / len(d) * 100).round(2))
     for n, d in tables.items()},
    names=["tabela", "coluna"],
)
null_report

nulos    pct
tabela        coluna                                     
orders        order_approved_at                160   0.16
              order_delivered_carrier_date    1783   1.79
              order_delivered_customer_date   2965   2.98
order_reviews review_comment_title           87656  88.34
              review_comment_message         58247  58.70
products      product_category_name            610   1.85
              product_name_lenght              610   1.85
              product_description_lenght       610   1.85
              product_photos_qty               610   1.85
              product_weight_g                   2   0.01
              product_length_cm                  2   0.01
              product_height_cm                  2   0.01
              product_width_cm                   2   0.01

nem todo nulo é um erro:

| Coluna | motivo | tratamento |
|---|---|---|
| `order_approved_at`, `order_delivered_*` | O pedido ainda não chegou nesse status ou foi cancelado | **Manter NULL** pois é informação |
| `review_comment_title/message` | O cliente só deu a nota | **Manter NULL** |
| `product_category_name` e outras 4 colunas (~610) | Anúncios sem metadados | Rótulo `unknown` na categoria; fotos continuam NULL |
| `product_weight_g` e dimensões (2) | Cadastro incompleto | Imputar pela mediana da categoria com flag |

### 3.2 Duplicatas

1. Linha inteira repetida -> apagar
2. Mesma chave, linhas diferentes -> pode ser erro ou pode ser informação

In [10]:
dups = pd.DataFrame({
    "linhas inteiras repetidas": {n: int(d.duplicated().sum()) for n, d in tables.items()},
})
key_cols = {
    "customers": ["customer_id"], "orders": ["order_id"], "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"], "order_reviews": ["review_id"],
    "products": ["product_id"], "sellers": ["seller_id"], "geolocation": ["geolocation_zip_code_prefix"],
    "category_translation": ["product_category_name"],
}
dups["chave que deveria ser única"] = pd.Series({n: " + ".join(k) for n, k in key_cols.items()})
dups["linhas com chave repetida"] = pd.Series({n: int(tables[n].duplicated(k).sum()) for n, k in key_cols.items()})
dups

,linhas inteiras repetidas,chave que deveria ser única,linhas com chave repetida
customers,0,customer_id,0
orders,0,order_id,0
order_items,0,order_id + order_item_id,0
order_payments,0,order_id + payment_sequential,0
order_reviews,0,review_id,814
products,0,product_id,0
sellers,0,seller_id,0
geolocation,261831,geolocation_zip_code_prefix,981148
category_translation,0,product_category_name,0


- `geolocation` 261 mil linhas 100% idênticas → a chave CEP se repete porque cada CEP tem vários pontos de GPS -> agregar
- `order_reviews` `review_id` repete 814 vezes

In [15]:
reviews = tables["order_reviews"]
repeated = reviews[reviews["review_id"].duplicated(keep=False)].sort_values("review_id")
print("review_id que aparece em mais de um pedido:", repeated["review_id"].nunique())
print("pedidos diferentes:", (repeated.groupby("review_id")["order_id"].nunique() > 1).all())
repeated[["review_id", "order_id", "review_score", "review_creation_date"]].head(6)

review_id que aparece em mais de um pedido: 789
pedidos diferentes: True


,review_id,order_id,review_score,review_creation_date
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,2018-03-07 00:00:00
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,2018-03-07 00:00:00
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,2017-09-21 00:00:00
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,2017-09-21 00:00:00
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,2018-03-07 00:00:00
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,2018-03-07 00:00:00


> `drop_duplicates(subset="review_id")` -> apaga avaliações válidas, a mesma avaliação pode ser associada a dois pedidos.
> A chave correta é o par `(review_id, order_id)`

## 4. Diagnóstico

In [16]:
problems = []   # (tabela, problema, linhas afetadas, o que a limpeza deve fazer)

def register(table, problem, rows, action):
    problems.append({"tabela": table, "problema": problem, "linhas afetadas": int(rows), "tratamento": action})
    return int(rows)

register("geolocation", "linhas 100% duplicadas", tables["geolocation"].duplicated().sum(), "remover (agregar por CEP)")
register("order_reviews", "review_id repetido em outros pedidos", reviews["review_id"].duplicated(keep=False).sum(), "manter; chave = (review_id, order_id)")

1603

### 4.1 CEP
CEP é um código, não um número: `01046` ≠ `1046`. Ao ler o CSV como número, os zeros à esquerda somem

In [19]:
from src.etl.cleaning import standardize_zip

cust = tables["customers"]
zip_raw = cust["customer_zip_code_prefix"]
print("Tipo pelo pandas:", zip_raw.dtype, "| menor valor:", zip_raw.min())

lost = zip_raw.astype(str).str.len() < 5
print(f"CEPs com menos de 5 dígitos: {lost.sum():,} ({lost.mean():.1%} dos clientes)")
pd.DataFrame({"original": zip_raw[lost].head(5), "tratado": standardize_zip(zip_raw[lost].head(5))})

Tipo pelo pandas: int64 | menor valor: 1003
CEPs com menos de 5 dígitos: 23,995 (24.1% dos clientes)


,original,tratado
1,9790,09790
2,1151,01151
3,8775,08775
6,4534,04534
13,5704,05704


In [20]:
for name, col in [("customers", "customer_zip_code_prefix"), ("sellers", "seller_zip_code_prefix"), ("geolocation", "geolocation_zip_code_prefix")]:
    n = (tables[name][col].astype(str).str.len() < 5).sum()
    register(name, "CEP sem o zero à esquerda", n, "restaurar para 5 dígitos (string)")
    print(f"{name:12s} {n:>9,} CEPs afetados")

customers       23,995 CEPs afetados
sellers          1,027 CEPs afetados
geolocation    245,733 CEPs afetados


Problema: ao cruzar com outra fonte onde o CEP é texto (`"01046"`) ocorre erro (joins, merge)
CEP geralmente é guardado como texto de 5 caracteres

### 4.2 Cidades

Cidade é texto digitado. Para fazer `JOIN`, agrupar e contar, grafias divergentes causam erro

In [22]:
from src.etl.cleaning import strip_accents

sellers = tables["sellers"]
noisy = sellers[sellers["seller_city"].str.contains(r"[@/\\,\d]|\s-\s|\s(?:sp|rj|mg|pr|rs|sc)$|^(?:sp|rj)$", regex=True, case=False)]
print(f"{len(noisy)} Exemplos com erro:")
noisy[["seller_city", "seller_state"]].drop_duplicates().head(20)

30 Exemplos com erro:


,seller_city,seller_state
78,lages - sc,SC
237,auriflama/sp,SP
246,sao paulo / sao paulo,SP
517,04482255,RJ
551,"novo hamburgo, rio grande do sul, brasil",RS
622,cariacica / es,ES
826,sao paulo - sp,SP
869,sbc/sp,SP
945,santo andre/sao paulo,SP
1004,sp / sp,SP


`04482255` (CEP) no campo cidade
sufixos como `/sp`, `, brasil`, `rj`

em `geolocation` -> variações de acento e caixa

In [23]:
geo = tables["geolocation"]
customers_city = tables["customers"]["customer_city"]

has_accent = geo["geolocation_city"].map(lambda v: v != strip_accents(v))
print(f"geolocation: {has_accent.mean():.1%} das linhas têm acento na cidade")
print(f"customers:   {(customers_city != customers_city.map(strip_accents)).mean():.1%} das linhas têm acento na cidade")

# a mesma cidade, com e sem acento, dentro da geolocation
spellings = (geo[["geolocation_city", "geolocation_state"]].drop_duplicates()
             .assign(chave=lambda d: d["geolocation_city"].map(lambda v: strip_accents(v).lower()))
             .groupby(["chave", "geolocation_state"])["geolocation_city"].agg(["nunique", lambda s: sorted(set(s))]))
multi = spellings[spellings["nunique"] > 1]
print(f"\n{len(multi):,} cidades escritas de mais de um jeito. Exemplos:")
multi.head(6)

geolocation: 7.3% das linhas têm acento na cidade
customers:   0.0% das linhas têm acento na cidade

2,109 cidades escritas de mais de um jeito. Exemplos:


,,nunique,<lambda_0>
chave,geolocation_state,,
abadiania,GO,2,"[abadiania, abadiânia]"
abaete,MG,2,"[abaete, abaeté]"
abaira,BA,2,"[abaira, abaíra]"
abare,BA,2,"[abare, abaré]"
abatia,PR,2,"[abatia, abatiá]"
acailandia,MA,2,"[acailandia, açailândia]"


`customers` não tem nenhum acento (`sao paulo`) e a `geolocation` mistura `são paulo` e `sao paulo` 
Um `JOIN` por nome de cidade iria perder linhas
definir regra de negócio (tadas minusculas ou maiusculas, inicial maiuscula e resto maiuscula - com ou sem acento)

In [24]:
register("geolocation", "cidade com acento/caixa diferente", has_accent.sum(), "padronizar (minúscula, sem acento)")
register("sellers", "cidade com lixo (e-mail, número, sufixo UF)", len(noisy), "limpar; se inválida, reparar pelo CEP")

30

### 4.3 UF × CEP

faixa de CEP por Estado -> definida pelos Correios (ex.: SP = 01000–19999, RJ = 20000–28999) -> teste de consistência

In [25]:
from src.etl.cleaning import zip_state, zip_state_mismatch, standardize_zip

for name, prefix in [("customers", "customer"), ("sellers", "seller")]:
    d = tables[name]
    mismatch = zip_state_mismatch(standardize_zip(d[f"{prefix}_zip_code_prefix"]), d[f"{prefix}_state"])
    register(name, "UF diverge da faixa de CEP", mismatch.sum(), "sinalizar (zip_state_mismatch)")
    print(f"{name:10s} UF diverge da faixa do CEP em {mismatch.sum():>4} linhas de {len(d):,}")

d = tables["sellers"]
m = zip_state_mismatch(standardize_zip(d["seller_zip_code_prefix"]), d["seller_state"])
d.loc[m, ["seller_zip_code_prefix", "seller_city", "seller_state"]].assign(
    UF_do_CEP=standardize_zip(d.loc[m, "seller_zip_code_prefix"]).map(zip_state)).head(6)

customers  UF diverge da faixa do CEP em    0 linhas de 99,441
sellers    UF diverge da faixa do CEP em   35 linhas de 3,095


,seller_zip_code_prefix,seller_city,seller_state,UF_do_CEP
70,95076,caxias do sul,SP,RS
114,21210,rio de janeiro,RN,RJ
198,85301,laranjeiras do sul,SP,PR
206,87360,goioere,SP,PR
275,86170,sertanopolis,SP,PR
311,85960,marechal candido rondon,SP,PR


Nos clientes a UF bate
em vendedores há erro -> pipeline sinaliza erro

### 4.4 Geolocalização

In [27]:
from src.etl.cleaning import BRAZIL_LAT_RANGE, BRAZIL_LNG_RANGE

outside = ~(geo["geolocation_lat"].between(*BRAZIL_LAT_RANGE) & geo["geolocation_lng"].between(*BRAZIL_LNG_RANGE))
print(f"Pontos fora do retângulo do Brasil: {outside.sum()}")
print("Latitude: ", geo["geolocation_lat"].min().round(1), "a", geo["geolocation_lat"].max().round(1), "(-33,8 a 5,3)")
print("Longitude:", geo["geolocation_lng"].min().round(1), "a", geo["geolocation_lng"].max().round(1), "(-74 a -34,7)")
geo.loc[outside, ["geolocation_zip_code_prefix", "geolocation_city", "geolocation_state", "geolocation_lat", "geolocation_lng"]].head(6)

Pontos fora do retângulo do Brasil: 42
Latitude:  -36.6 a 45.1 (-33,8 a 5,3)
Longitude: -101.5 a 121.1 (-74 a -34,7)


,geolocation_zip_code_prefix,geolocation_city,geolocation_state,geolocation_lat,geolocation_lng
387565,18243,bom retiro da esperanca,SP,28.008978,-15.536867
513631,28165,vila nova de campos,RJ,41.614052,-8.411675
513643,28155,santa maria,RJ,-34.586422,-58.732101
513754,28155,santa maria,RJ,42.439286,13.820214
514429,28333,raposo,RJ,38.381672,-6.328200
516682,28595,portela,RJ,43.684961,-7.411080


In [ ]:
import plotly.express as px

sample = geo.sample(
    30000, # amostra de 30 K
    random_state=1
    ).assign(fora_do_brasil=lambda d: outside.loc[d.index])
fig = px.scatter(
    sample, x="geolocation_lng", y="geolocation_lat", color="fora_do_brasil", opacity=0.5, height=430,
    color_discrete_map={
        False: "#2a78d6",   # BR
        True: "#e34948"     # gringo
        }, title="Pontos de GPS da geolocation")
fig.update_traces(marker_size=4)
fig.show()
register("geolocation", "coordenadas fora do Brasil", outside.sum(), "descartar antes de calcular o centro do CEP")

42

In [30]:
# Um CEP de SP com latitude na Europa distorce a média do CEP
# alguns CEPs aparecem em mais de uma localidade
states_per_zip = geo.groupby("geolocation_zip_code_prefix")["geolocation_state"].nunique()
print(f"CEPs associados a mais de uma UF: {(states_per_zip > 1).sum()}")
register("geolocation", "CEP em mais de uma UF", (states_per_zip > 1).sum(), "ficar com a UF mais frequente")
dispersion = geo.groupby("geolocation_zip_code_prefix")["geolocation_lat"].std()
print(f"CEPs com pontos muito dispersos (desvio): {(dispersion > 1).sum()}")

CEPs associados a mais de uma UF: 8
CEPs com pontos muito dispersos (desvio): 111


### 4.5 Categorias de produto

In [32]:
products, translation = tables["products"], tables["category_translation"]

print("Produtos sem categoria:", products["product_category_name"].isna().sum())
missing_translation = sorted(set(products["product_category_name"].dropna()) - set(translation["product_category_name"]))
print("Categorias que existem nos produtos mas NÃO na tabela de tradução:", missing_translation)
print("Produtos afetados:", products["product_category_name"].isin(missing_translation).sum())
print("\nTotal de categorias distintas:", products["product_category_name"].nunique())

Produtos sem categoria: 610
Categorias que existem nos produtos mas NÃO na tabela de tradução: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Produtos afetados: 13

Total de categorias distintas: 73


In [35]:
register("products", "produto sem categoria", products["product_category_name"].isna().sum(), "rótulo 'unknown'")
register("products", "categoria sem tradução no arquivo do Kaggle", products["product_category_name"].isin(missing_translation).sum(), "tradução manual (fallback)")
products["product_category_name"].value_counts().head(10)
# agrupar essas categorias em macro-categoria para facilitar a leitura

product_category_name
cama_mesa_banho           3029
esporte_lazer             2867
moveis_decoracao          2657
beleza_saude              2444
utilidades_domesticas     2335
automotivo                1900
informatica_acessorios    1639
brinquedos                1411
relogios_presentes        1329
telefonia                 1134
Name: count, dtype: int64

### 4.6 Números: zeros impossíveis e outliers

In [34]:
size_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm", "product_photos_qty"]
products[size_cols].describe().T[["count", "min", "50%", "max"]].assign(zeros=(products[size_cols] == 0).sum())

,count,min,50%,max,zeros
product_weight_g,32949.0,0.0,700.0,40425.0,4
product_length_cm,32949.0,7.0,25.0,105.0,0
product_height_cm,32949.0,2.0,13.0,105.0,0
product_width_cm,32949.0,6.0,20.0,118.0,0
product_photos_qty,32341.0,1.0,1.0,20.0,0


In [37]:
n_zero_weight = (products["product_weight_g"] == 0).sum()
register("products", "peso = 0 g", n_zero_weight, "anular e imputar pela mediana da categoria")
print(f"{n_zero_weight} produtos com peso zero")
products.loc[products["product_weight_g"] == 0, ["product_id", "product_category_name", "product_weight_g"]]

4 produtos com peso zero


,product_id,product_category_name,product_weight_g
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,0.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,0.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,0.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,0.0


Preços

In [45]:
import numpy as np
# outlier?
# LI = Q1 - 1,5 * IQR
# LS = Q3 + 1,5 * IQR
# outlier = valor < limite inferior (LI) ou valor > limite superior (LS)
# valor acima de `Q3 + 3×IQR`.

price = items["price"]
q1, q3 = price.quantile([0.25, 0.75])
fence = q3 + 3 * (q3 - q1)
print(f"Mediana R$ {price.median():.2f}")
print(f"Média R$ {price.mean():.2f}")
print(f"Máximo R$ {price.max():,.2f}")
print(f"Cerca de outlier extremo (Q3 + 3×IQR): R$ {fence:.2f} → {(price > fence).sum():,} itens ({(price > fence).mean():.1%})")

fig = px.histogram(items, x="price", nbins=80, log_y=True, height=340, title="Distribuição de preços (eixo Y em escala log)")
fig.add_vline(x=fence, line_dash="dash", line_color="#eb6834", annotation_text="Q3 + 3×IQR")
fig.show()
# Um produto de R$ 6K é real?

Mediana R$ 74.99
Média R$ 120.65
Máximo R$ 6,735.00
Cerca de outlier extremo (Q3 + 3×IQR): R$ 419.90 → 4,074 itens (3.6%)


In [46]:
freight_above = (items["freight_value"] > items["price"]).sum()
register("order_items", "preço extremo (> Q3 + 3×IQR)", (price > fence).sum(), "manter e sinalizar (is_price_outlier)")
register("order_items", "frete maior que o preço do item", freight_above, "sinalizar (is_freight_above_price)")
print(f"Itens com frete maior que o preço: {freight_above:,} | com frete zero: {(items['freight_value'] == 0).sum()}")

Itens com frete maior que o preço: 4,124 | com frete zero: 383


### 4.6b Sanidade temporal

### 4.7 Datas
Todas as datas chegam como texto.
Lógica: `compra ≤ aprovação ≤ postagem na transportadora ≤ entrega ao cliente`.

In [47]:
tcols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
         "order_delivered_customer_date", "order_estimated_delivery_date"]
o = orders.copy()
print("Tipo original:", o["order_purchase_timestamp"].dtype)
for c in tcols:
    o[c] = pd.to_datetime(o[c], errors="coerce")

checks = {
    "aprovação antes da compra": o["order_approved_at"] < o["order_purchase_timestamp"],
    "postagem antes da compra": o["order_delivered_carrier_date"] < o["order_purchase_timestamp"],
    "postagem antes da aprovação": o["order_delivered_carrier_date"] < o["order_approved_at"],
    "entrega antes da postagem": o["order_delivered_customer_date"] < o["order_delivered_carrier_date"],
    "'delivered' sem data de entrega": (o["order_status"] == "delivered") & o["order_delivered_customer_date"].isna(),
    "não 'delivered' COM data de entrega": (o["order_status"] != "delivered") & o["order_delivered_customer_date"].notna(),
}
date_report = pd.Series({k: int(v.sum()) for k, v in checks.items()}, name="pedidos")
for k, v in date_report.items():
    if v:
        register("orders", k, v, "sinalizar (has_date_anomaly), não apagar")
date_report.to_frame()

Tipo original: object


,pedidos
aprovação antes da compra,0
postagem antes da compra,166
postagem antes da aprovação,1359
entrega antes da postagem,23
'delivered' sem data de entrega,8
não 'delivered' COM data de entrega,6


postagem antes da aprovação vendedor marca o envio antes de o pagamento ser confirmado
Erro, mas é comportamento do negócio

In [48]:
limit = pd.to_datetime(items["shipping_limit_date"], errors="coerce")
print("Ano da data limite de envio:")
print(limit.dt.year.value_counts().sort_index().to_string())
register("order_items", "data limite de envio em 2020 (pedidos de 2016-2018)", (limit.dt.year >= 2020).sum(), "anular (fora da janela plausível)")

Ano da data limite de envio:
shipping_limit_date
2016      370
2017    49765
2018    62511
2020        4


4

### 4.8 Pagamentos

In [49]:
pay = tables["order_payments"]
print(pay["payment_type"].value_counts().to_string())
print("\nParcelas = 0:", (pay["payment_installments"] == 0).sum(), "→", pay.loc[pay["payment_installments"] == 0, "payment_type"].unique())
pay[pay["payment_type"] == "not_defined"]

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3

Parcelas = 0: 2 → ['credit_card']


,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [50]:
register("order_payments", "tipo 'not_defined' (valor zero)", (pay["payment_type"] == "not_defined").sum(), "sinalizar como inválido (is_valid_payment)")
register("order_payments", "cartão com 0 parcelas", (pay["payment_installments"] == 0).sum(), "ajustar para 1")

# o total pago = itens + frete
items_total = (items["price"] + items["freight_value"]).groupby(items["order_id"]).sum()
paid_total = pay.groupby("order_id")["payment_value"].sum()
both = pd.concat([items_total.rename("itens+frete"), paid_total.rename("pago")], axis=1).dropna()
diff = (both["pago"] - both["itens+frete"]).abs()
register("cross_table", "total pago difere de itens + frete (> R$ 1)", (diff > 1).sum(), "sinalizar (juros/desconto/voucher)")
print(f"Pedidos onde o total pago difere de itens+frete em mais de R$ 1: {(diff > 1).sum()} de {len(both):,}")
both.assign(dif=(both['pago'] - both['itens+frete']).round(2)).loc[diff.sort_values(ascending=False).head(5).index]

Pedidos onde o total pago difere de itens+frete em mais de R$ 1: 249 de 98,665


,itens+frete,pago,dif
order_id,,,
ce6d150fb29ada17d2082f4847107665,1403.66,1586.47,182.81
6e5fe7366a2e1bfbf3257dba0af1267f,287.91,406.92,119.01
70b742795bc441e94a44a084b6d9ce7a,466.93,578.82,111.89
996c7e73600ad3723e8627ab7bef81e4,587.90,664.43,76.53
70b7e94ea46d3e8b5bc12a50186edaf0,213.15,274.84,61.69


### 4.9 Avaliações

O comentário é texto livre escrito por clientes

In [51]:
msg = reviews["review_comment_message"].dropna().astype(str)
report = pd.Series({
    "comentários": len(msg),
    "com quebra de linha (\\r\\n)": msg.str.contains(r"[\r\n]").sum(),
    "com espaços duplicados": msg.str.contains(r"\s{2,}").sum(),
    "com espaço no começo/fim": (msg != msg.str.strip()).sum(),
    "muito curtos (≤ 3 caracteres)": (msg.str.len() <= 3).sum(),
    "só pontuação/números ('.', '10')": msg.str.fullmatch(r"[\W\d_]+").sum(),
    "TODO EM MAIÚSCULAS (> 10 letras)": msg.map(lambda x: len(x) > 10 and x == x.upper() and x != x.lower()).sum(),
    "em branco": (msg.str.strip() == "").sum(),
}, name="quantidade")
report.to_frame()

,quantidade
comentários,40977
com quebra de linha (\r\n),3852
com espaços duplicados,3852
com espaço no começo/fim,9451
muito curtos (≤ 3 caracteres),777
"só pontuação/números ('.', '10')",186
TODO EM MAIÚSCULAS (> 10 letras),1120
em branco,27


In [52]:
register("order_reviews", "comentário com quebra de linha/espaços extras", (msg != msg.str.strip()).sum() + msg.str.contains(r"[\r\n]").sum(), "normalizar espaços (mantém acento e caixa)")
register("order_reviews", "comentário de baixa informação (≤ 3 caracteres ou só símbolos)", (msg.str.len() <= 3).sum() + msg.str.fullmatch(r"[\W\d_]+").sum(), "sinalizar (is_low_information)")
print("Exemplos de comentários curtos mais comuns:", msg[msg.str.len() <= 3].value_counts().head(6).to_dict())
print("\nExemplo de comentário com quebra de linha:")
print(repr(msg[msg.str.contains(r"[\r\n]")].iloc[0][:160]))

Exemplos de comentários curtos mais comuns: {'Bom': 189, 'bom': 107, 'Ok': 72, 'ok': 55, '.': 51, 'Boa': 47}

Exemplo de comentário com quebra de linha:
'Mas um pouco ,travando...pelo valor ta Boa.\r\n'


para análise de sentimento só remove quebras de linha, espaços extras e codificação quebrada
`review_comment_message_norm` -> versão normalizada para PLN

### 4.10 Integridade referencial

In [53]:
integrity = pd.Series({
    "itens → pedido inexistente": (~items["order_id"].isin(orders["order_id"])).sum(),
    "itens → produto inexistente": (~items["product_id"].isin(products["product_id"])).sum(),
    "itens → vendedor inexistente": (~items["seller_id"].isin(sellers["seller_id"])).sum(),
    "pedidos → cliente inexistente": (~orders["customer_id"].isin(customers["customer_id"])).sum(),
    "pagamentos → pedido inexistente": (~pay["order_id"].isin(orders["order_id"])).sum(),
    "avaliações → pedido inexistente": (~reviews["order_id"].isin(orders["order_id"])).sum(),
}, name="órfãos")
print(integrity.to_string())

no_items = ~orders["order_id"].isin(items["order_id"])
print(f"\nPedidos SEM nenhum item: {no_items.sum()} →", orders.loc[no_items, "order_status"].value_counts().to_dict())
register("cross_table", "pedidos sem nenhum item", no_items.sum(), "manter (cancelados/indisponíveis), sem receita")

itens → pedido inexistente         0
itens → produto inexistente        0
itens → vendedor inexistente       0
pedidos → cliente inexistente      0
pagamentos → pedido inexistente    0
avaliações → pedido inexistente    0

Pedidos SEM nenhum item: 775 → {'unavailable': 603, 'canceled': 164, 'created': 5, 'invoiced': 2, 'shipped': 1}


775

Os pedidos sem item são `unavailable`/`canceled` ficam na base mas sem receita

## 5. Diagnostico

In [54]:
scoreboard = pd.DataFrame(problems).sort_values(["tabela", "linhas afetadas"], ascending=[True, False]).reset_index(drop=True)
scoreboard["linhas afetadas"] = scoreboard["linhas afetadas"].map("{:,}".format)
scoreboard

,tabela,problema,linhas afetadas,tratamento
0,cross_table,pedidos sem nenhum item,775,"manter (cancelados/indisponíveis), sem receita"
1,cross_table,total pago difere de itens + frete (> R$ 1),249,sinalizar (juros/desconto/voucher)
2,customers,CEP sem o zero à esquerda,"23,995",restaurar para 5 dígitos (string)
3,customers,UF diverge da faixa de CEP,0,sinalizar (zip_state_mismatch)
4,geolocation,linhas 100% duplicadas,"261,831",remover (agregar por CEP)
5,geolocation,CEP sem o zero à esquerda,"245,733",restaurar para 5 dígitos (string)
6,geolocation,cidade com acento/caixa diferente,"73,441","padronizar (minúscula, sem acento)"
7,geolocation,coordenadas fora do Brasil,42,descartar antes de calcular o centro do CEP
8,geolocation,CEP em mais de uma UF,8,ficar com a UF mais frequente
9,geolocation,CEP em mais de uma UF,8,ficar com a UF mais frequente
